# SPY 0DTE Directional Backtest (Polygon)

Backtest a simple **0DTE SPY momentum** strategy on real Polygon options data over the last ~2 years.

## Strategy
- Look at the **first 5-minute candle** of the regular session (09:30–09:35 ET).
- **Green candle** (close ≥ open) → **buy a 0DTE call, 1 strike ITM**.
- **Red candle** (close < open) → **buy a 0DTE put, 1 strike ITM**.
- Enter at **09:35 ET** at the option's price.
- Immediately place a **bracket (OCO) order**: a **take-profit limit at +5%** of premium and a **stop at −20%** of premium. We do **not** actively manage the position after that — whichever resting order fills first wins.
- If neither fills, **close at end of day** (~15:55 ET).
- **One trade per day, every trading day.** Deploy the **full account balance** each day, capped at **$50,000** of capital deployed.
- Starting balance: **$1,000**.

## Costs modeled
- **Commission:** $0.65 per contract, charged on **both** the entry and the exit.
- **Slippage:** the option must trade **$0.01 past** the take-profit price for that limit order to fill (the profit fills at the +5% price, but only once price reaches +5% + 1¢).

## Reported metrics
Final balance, avg trade PnL ($ and %), number of losers, trades-to-$50k, positive months, plus win rate, max drawdown, profit factor, streaks, CAGR, total commissions, and an equity curve + monthly PnL chart.

---
### ⚠️ Notes
- Requires a Polygon **Options** subscription with minute aggregates + 2yr history (you have the top tier — you're covered).
- Fills are modeled at the resting order prices. Real 0DTE bid/ask spreads still introduce risk beyond the 1¢ slippage assumed here, so treat results as optimistic.
- Betting the **entire account on one 0DTE option** is extremely high variance / high risk of ruin. Research only, not trading advice.

In [ ]:
# If running in Colab these are already installed; harmless to ensure.
# !pip -q install requests pandas numpy matplotlib
import requests, time, math
import datetime as dt
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

## 1. Configuration
Paste your Polygon API key. Adjust the date range, strategy parameters and costs here.

In [ ]:
# ---- API ----
POLYGON_API_KEY = "YOUR_POLYGON_API_KEY"   # <-- paste your Polygon key here in Colab (do NOT commit it)

# ---- Date range (last ~2 years) ----
START_DATE = dt.date(2024, 6, 1)
END_DATE   = dt.date(2026, 6, 1)

# ---- Strategy parameters ----
START_BALANCE = 1_000.0     # starting account balance
MAX_DEPLOY    = 50_000.0    # cap on capital deployed per trade
PROFIT_TARGET = 0.05        # +5% on option premium
STOP_LOSS     = 0.20        # -20% on option premium
ITM_STEPS     = 1           # how many strikes in-the-money (1 = first ITM strike)
STRIKE_STEP   = 1.0         # SPY listed strikes are $1 apart near the money

# ---- Costs ----
COMMISSION = 0.65           # $ per contract, charged on BOTH entry and exit
SLIPPAGE   = 0.01           # extra $ the option must move past target to fill the profit order

ENTRY_TIME = "09:35"        # enter right after the first 5-min candle
EOD_TIME   = "15:55"        # force close before the bell

# ---- API politeness ----
# Unlimited plans: 0.0 is fine. Free/low tiers: raise (e.g. 12.5 for 5 req/min).
REQUEST_SLEEP = 0.0

assert POLYGON_API_KEY != "YOUR_POLYGON_API_KEY", "Set your Polygon API key first!"
print(f"Backtesting {START_DATE} -> {END_DATE}")

## 2. Polygon helpers

In [ ]:
BASE = "https://api.polygon.io"
SESSION = requests.Session()

def _get(url, params=None, max_retries=6):
    params = dict(params or {})
    params["apiKey"] = POLYGON_API_KEY
    backoff = 1.0
    for attempt in range(max_retries):
        try:
            r = SESSION.get(url, params=params, timeout=30)
        except requests.RequestException:
            time.sleep(backoff); backoff *= 2; continue
        if r.status_code == 200:
            if REQUEST_SLEEP:
                time.sleep(REQUEST_SLEEP)
            return r.json()
        if r.status_code in (401, 403):
            raise RuntimeError(
                f"{r.status_code} from Polygon: {r.text[:200]}\\n"
                "-> Key lacks OPTIONS data access for this request."
            )
        if r.status_code == 429:        # rate limited
            time.sleep(backoff); backoff *= 2; continue
        # transient 5xx etc.
        time.sleep(backoff); backoff *= 2
    raise RuntimeError(f"Failed after {max_retries} retries: {url}")

def get_aggs(ticker, mult, timespan, date_str):
    """Return a tz-aware (America/New_York) OHLCV DataFrame, or None if no data."""
    url = f"{BASE}/v2/aggs/ticker/{ticker}/range/{mult}/{timespan}/{date_str}/{date_str}"
    j = _get(url, {"adjusted": "true", "sort": "asc", "limit": 50000})
    res = j.get("results")
    if not res:
        return None
    df = pd.DataFrame(res).rename(
        columns={"o": "open", "h": "high", "l": "low", "c": "close", "v": "vol"})
    df["t"] = (pd.to_datetime(df["t"], unit="ms", utc=True)
                 .dt.tz_convert("America/New_York"))
    return df.set_index("t").sort_index()

def get_trading_days(start, end):
    """Use SPY daily bars: one bar == one trading day (handles holidays)."""
    url = f"{BASE}/v2/aggs/ticker/SPY/range/1/day/{start}/{end}"
    j = _get(url, {"adjusted": "true", "sort": "asc", "limit": 50000})
    out = []
    for r in j.get("results", []):
        d = pd.to_datetime(r["t"], unit="ms", utc=True).tz_convert("America/New_York").date()
        out.append(d)
    return out

def occ_ticker(expiry, cp, strike):
    """Polygon/OCC option symbol, e.g. O:SPY240617C00580000 (0DTE => expiry == trade date)."""
    return f"O:SPY{expiry:%y%m%d}{cp}{int(round(strike * 1000)):08d}"

def pick_contract(first_open, first_close):
    """Green -> call ITM (strike below spot); Red -> put ITM (strike above spot)."""
    spot = first_close
    if first_close >= first_open:                      # GREEN -> CALL
        cp = "C"
        strike = math.floor(spot)                      # first ITM strike for a call
        if strike >= spot:
            strike -= STRIKE_STEP
        strike -= (ITM_STEPS - 1) * STRIKE_STEP
    else:                                              # RED -> PUT
        cp = "P"
        strike = math.ceil(spot)                       # first ITM strike for a put
        if strike <= spot:
            strike += STRIKE_STEP
        strike += (ITM_STEPS - 1) * STRIKE_STEP
    return cp, float(strike)

## 3. Run the backtest
For each trading day: read the first 5-min SPY candle → pick the 0DTE contract → buy at 09:35 → place a resting **take-profit (+5%, +1¢ slippage)** and **stop (−20%)** bracket → settle at whichever fills first, else close at EOD.

Resting orders are *not* re-evaluated minute by minute — we only scan the later 1-min bars to determine **which bracket order fills first** (a backtest needs intraday granularity to know the sequence; it does not imply active management).

~2 API calls per trading day (≈1,000 calls over 2 years).

In [ ]:
def simulate_day(d):
    """Return a trade dict for date d, or None if the day is skipped."""
    ds = d.isoformat()

    # --- signal: first 5-min SPY candle ---
    spy = get_aggs("SPY", 5, "minute", ds)
    if spy is None:
        return None
    spy = spy.between_time("09:30", "16:00")
    if spy.empty:
        return None
    first = spy.iloc[0]
    cp, strike = pick_contract(first["open"], first["close"])
    spot  = float(first["close"])
    color = "green" if first["close"] >= first["open"] else "red"

    # --- option 1-min bars for the 0DTE contract ---
    ticker = occ_ticker(d, cp, strike)
    opt = get_aggs(ticker, 1, "minute", ds)
    if opt is None:
        return None
    opt = opt.between_time(ENTRY_TIME, EOD_TIME)
    if opt.empty:
        return None

    entry = float(opt.iloc[0]["open"])           # buy at 09:35 bar open
    if not entry or entry <= 0:
        return None

    # Bracket placed at entry: take-profit limit + stop. Whichever fills first wins.
    target_fill    = entry * (1 + PROFIT_TARGET)            # profit order fills here
    target_trigger = target_fill + SLIPPAGE                # ...but needs +1c to fill
    stop_fill      = entry * (1 - STOP_LOSS)               # stop order fills here

    exit_price = exit_reason = exit_time = None
    for ts, bar in opt.iterrows():
        hit_t = bar["high"] >= target_trigger
        hit_s = bar["low"]  <= stop_fill
        if hit_t and hit_s:                 # both touched in one bar -> assume worst (stop)
            exit_price, exit_reason = stop_fill, "stop(ambiguous)"
        elif hit_t:
            exit_price, exit_reason = target_fill, "target"
        elif hit_s:
            exit_price, exit_reason = stop_fill, "stop"
        if exit_price is not None:
            exit_time = ts
            break
    if exit_price is None:                  # neither bracket filled -> close at EOD
        exit_price, exit_reason, exit_time = float(opt.iloc[-1]["close"]), "eod", opt.index[-1]

    return dict(date=d, color=color, type=cp, strike=strike, spot=spot,
                ticker=ticker, entry=entry, exit=exit_price, reason=exit_reason,
                exit_time=exit_time)


days = get_trading_days(START_DATE, END_DATE)
print(f"{len(days)} trading days: {days[0]} -> {days[-1]}\\n")

trades, balance, skipped = [], START_BALANCE, 0
for i, d in enumerate(days, 1):
    try:
        t = simulate_day(d)
    except RuntimeError as e:
        if "OPTIONS data access" in str(e):
            raise
        t = None
    if t is None:
        skipped += 1
        continue

    deploy   = min(balance, MAX_DEPLOY)
    per_ctc  = t["entry"] * 100 + COMMISSION          # premium + entry commission
    contracts = int(deploy // per_ctc)                 # whole contracts only
    if contracts < 1:                                   # can't afford one contract
        skipped += 1
        continue

    gross      = contracts * 100 * (t["exit"] - t["entry"])
    commission = contracts * COMMISSION * 2            # entry + exit
    pnl        = gross - commission
    cost       = contracts * t["entry"] * 100
    balance   += pnl
    t.update(contracts=contracts, gross=gross, commission=commission,
             pnl=pnl, ret=pnl / cost, balance=balance)
    trades.append(t)

    if len(trades) % 25 == 0:
        print(f"{d}  trades={len(trades):4d}  balance=${balance:,.0f}")

td = pd.DataFrame(trades)
print(f"\\nDone. {len(td)} trades, {skipped} days skipped (no data / unaffordable).")

## 4. Performance report

In [ ]:
def report(td):
    if td.empty:
        print("No trades were executed."); return

    n        = len(td)
    final    = td["balance"].iloc[-1]
    total_ret= final / START_BALANCE - 1
    wins     = td[td.pnl > 0]
    losses   = td[td.pnl <= 0]
    n_loss   = len(losses)
    win_rate = len(wins) / n

    avg_pnl_usd = td.pnl.mean()
    avg_ret_pct = td.ret.mean()
    avg_win = wins.pnl.mean() if len(wins) else 0.0
    avg_los = losses.pnl.mean() if n_loss else 0.0
    pf = wins.pnl.sum() / abs(losses.pnl.sum()) if losses.pnl.sum() != 0 else float("inf")
    total_comm = td.commission.sum()

    # trades to first reach $50k
    hit = td.index[td.balance >= 50_000]
    to50k = int(hit[0]) + 1 if len(hit) else None

    # monthly PnL
    m = td.copy()
    m["ym"] = pd.to_datetime(m.date).dt.to_period("M")
    monthly = m.groupby("ym").pnl.sum()
    pos_months = int((monthly > 0).sum())

    # max drawdown on the equity curve
    eq   = np.concatenate([[START_BALANCE], td.balance.values])
    peak = np.maximum.accumulate(eq)
    maxdd = float(((eq - peak) / peak).min())

    # win/loss streaks
    def longest(mask):
        best = cur = 0
        for v in mask:
            cur = cur + 1 if v else 0
            best = max(best, cur)
        return best
    win_streak  = longest((td.pnl > 0).tolist())
    loss_streak = longest((td.pnl <= 0).tolist())

    # CAGR
    days_span = (td.date.iloc[-1] - td.date.iloc[0]).days or 1
    cagr = (final / START_BALANCE) ** (365 / days_span) - 1

    reasons = td.reason.value_counts().to_dict()

    print("=" * 56)
    print("  SPY 0DTE DIRECTIONAL BACKTEST — RESULTS")
    print("=" * 56)
    print(f"  Period                 {td.date.iloc[0]} -> {td.date.iloc[-1]}")
    print(f"  Trades taken           {n}")
    print(f"  Starting balance       ${START_BALANCE:,.2f}")
    print(f"  FINAL BALANCE          ${final:,.2f}")
    print(f"  Total return           {total_ret*100:,.1f}%")
    print(f"  CAGR                   {cagr*100:,.1f}%")
    print("-" * 56)
    print(f"  Avg trade PnL          ${avg_pnl_usd:,.2f}   ({avg_ret_pct*100:+.2f}%)")
    print(f"  Avg winner / loser     ${avg_win:,.2f} / ${avg_los:,.2f}")
    print(f"  Win rate               {win_rate*100:.1f}%  ({len(wins)} W / {n_loss} L)")
    print(f"  Number of losers       {n_loss}")
    print(f"  Profit factor          {pf:.2f}")
    print(f"  Total commissions      ${total_comm:,.2f}")
    print("-" * 56)
    if to50k is not None:
        print(f"  Trades to reach $50k   {to50k}  (on {td.date.iloc[to50k-1]})")
    else:
        print(f"  Trades to reach $50k   never reached (peak ${td.balance.max():,.0f})")
    print(f"  Positive months        {pos_months} / {len(monthly)}")
    print(f"  Max drawdown           {maxdd*100:.1f}%")
    print(f"  Longest win streak     {win_streak}")
    print(f"  Longest loss streak    {loss_streak}")
    print(f"  Best / worst trade      ${td.pnl.max():,.2f} / ${td.pnl.min():,.2f}")
    print(f"  Exit breakdown         {reasons}")
    print(f"  Signal split           green/calls={int((td.color=='green').sum())}, "
          f"red/puts={int((td.color=='red').sum())}")
    print("=" * 56)

    return monthly

monthly = report(td)

## 5. Charts — equity curve & monthly PnL

In [ ]:
if not td.empty:
    fig, ax = plt.subplots(1, 2, figsize=(15, 5))

    # Equity curve (log scale handles 1k -> 50k+ growth)
    eq_dates = [td.date.iloc[0]] + list(td.date)
    eq_vals  = [START_BALANCE] + list(td.balance)
    ax[0].plot(eq_dates, eq_vals)
    ax[0].axhline(50_000, ls="--", lw=1, color="grey", label="$50k")
    ax[0].set_yscale("log")
    ax[0].set_title("Equity curve (log scale)")
    ax[0].set_ylabel("Balance ($)")
    ax[0].legend()
    ax[0].grid(True, which="both", alpha=0.3)

    # Monthly PnL
    if monthly is not None:
        colors = ["#2ca02c" if v > 0 else "#d62728" for v in monthly.values]
        ax[1].bar([str(p) for p in monthly.index], monthly.values, color=colors)
        ax[1].set_title("Monthly PnL ($)")
        ax[1].tick_params(axis="x", rotation=90)
        ax[1].grid(True, axis="y", alpha=0.3)

    plt.tight_layout()
    plt.show()

## 6. Trade log
Full trade-by-trade detail. Uncomment the CSV line to export.

In [ ]:
if not td.empty:
    show = td[["date","color","type","strike","spot","entry","exit","reason",
               "contracts","gross","commission","pnl","ret","balance"]].copy()
    show["ret"] = (show["ret"] * 100).round(2)
    for c in ["spot","entry","exit","gross","commission","pnl","balance"]:
        show[c] = show[c].round(2)
    # show.to_csv("spy_0dte_trades.csv", index=False)
    display(show.tail(30))